In [2]:
import os 


In [3]:
%pwd

'd:\\Text-summarizer\\research'

In [4]:
os.chdir("../")

In [5]:
%pwd

'd:\\Text-summarizer'

In [6]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [7]:
pip install -e .


Obtaining file:///D:/Text-summarizer
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Attempting uninstall: Text-summarizer
    Found existing installation: Text-summarizer 0.0.0
    Uninstalling Text-summarizer-0.0.0:
      Successfully uninstalled Text-summarizer-0.0.0
  Running setup.py develop for Text-summarizer
Note: you may need to restart the kernel to use updated packages.


  DEPRECATION: Legacy editable install of Text-summarizer==0.0.0 from file:///D:/Text-summarizer (setup.py develop) is deprecated. pip 25.0 will enforce this behaviour change. A possible replacement is to add a pyproject.toml or enable --use-pep517, and use setuptools >= 64. If the resulting installation is not behaving as expected, try using --config-settings editable_mode=compat. Please consult the setuptools documentation for more information. Discussion can be found at https://github.com/pypa/pip/issues/11457


In [8]:
from text_summarizer.constants import *
from text_summarizer.utils.common import read_yaml, create_directories

In [9]:
class ConfigurationManager:
    def __init__(self, config_file_path = CONFIG_FILE_PATH, params_file_path = PARAMS_FILE_PATH):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        # ADD THIS LINE BELOW
        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir = config.root_dir,
            source_URL = config.source_URL,
            local_data_file = config.local_data_file,
            unzip_dir = config.unzip_dir
        )

        return data_ingestion_config

In [10]:
import os
import urllib.request as request
import zipfile
from text_summarizer.logging import logger
from text_summarizer.utils.common import get_size

In [11]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} downloaded with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")


    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)


In [12]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-03-20 20:16:48,197: INFO: common]: yaml file: config\config.yaml loaded successfully
[2026-03-20 20:16:48,200: INFO: common]: yaml file: params.yaml loaded successfully
[2026-03-20 20:16:48,201: INFO: common]: created directory at: artifacts
[2026-03-20 20:16:48,203: INFO: common]: created directory at: artifacts/data_ingestion
[2026-03-20 20:16:48,214: INFO: 787099721]: File already exists of size: ~ 7718 KB
